 # Walmart Weekly Sales Forecasting
 ### Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np

# Load datasets
train = pd.read_csv("./data/train.csv")
stores = pd.read_csv("./data/stores.csv")
features = pd.read_csv("./data/features.csv")

# Left join operations for creating the main table
data = pd.merge(train, stores, on="Store", how="left")
data = pd.merge(data, features, on=["Store", "Date"], how="left")

# Deleting duplicate IsHoliday and renaming the other one
data = data.drop(columns="IsHoliday_y")
data = data.rename(columns={"IsHoliday_x": "IsHoliday"})

# Fill NaN values in markdown features
markdown_columns = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
data[markdown_columns] = data[markdown_columns].fillna(0)

# Filter the data (only Store 1)
data = data[data['Store'] == 1].reset_index(drop=True)

# Collapse all departments into 1 row per week
feature_rules = {
    'Weekly_Sales': 'sum',
    'IsHoliday': 'max',
    'Temperature': 'mean',
    'Fuel_Price': 'mean',
    'CPI': 'mean',
    'Unemployment': 'mean',
    'MarkDown1': 'mean',
    'MarkDown2': 'mean',
    'MarkDown3': 'mean',
    'MarkDown4': 'mean',
    'MarkDown5': 'mean'
}
data = data.groupby('Date').agg(feature_rules).reset_index()

 ### Feature Engineering

In [ ]:
# Convert date (string to datetime)
data['Date'] = pd.to_datetime(data['Date'])
# New features: Year, Month, Week of Year
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Week_of_Year'] = data['Date'].dt.isocalendar().week.astype(int)
# Convert IsHoliday (True/False to 1/0)
data['IsHoliday'] = data['IsHoliday'].astype(int)

# Sort the data by Date
data = data.sort_values(by='Date').reset_index(drop=True)
# New feature: Last Week Sales
data['Last_Week_Sales'] = data['Weekly_Sales'].shift(1)
data['Last_Week_Sales'] = data['Last_Week_Sales'].fillna(0)

 ### Data Splitting

In [ ]:
# Split data as train and test (05-02-2010 - 26-10-2012)
test_filter = (data['Year'] >= 2012) & (data['Month'] >= 10)
train_data = data[~test_filter].reset_index(drop=True)
val_data = data[test_filter].reset_index(drop=True)

# Target
y_train = train_data['Weekly_Sales']
# Train features
X_train = train_data.drop(columns=['Weekly_Sales', 'Date'], errors='ignore')
# Target validation
y_val = val_data['Weekly_Sales']
# Test
X_val = val_data.drop(columns=['Weekly_Sales', 'Date'], errors='ignore')

 ### Feature Selection, Training and Validation

In [ ]:
import xgboost as xgb
from sklearn.feature_selection import RFE, SequentialFeatureSelector, SelectFromModel
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.metrics import mean_squared_error
from mrmr import mrmr_regression

# Main model
main_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)

# WMAPE function
def wmape(real, prediction):
    return (np.abs(real - prediction).sum() / real.sum()) * 100

# Global tracking structures
all_results = []
best_predictions = {}
best_wmape_tracker = {method: float('inf') for method in ["MRMR", "RFE", "LASSO", "SFS"]}

# Helper function to train, predict and evaluate
def evaluate_selector(method_name, cols, k):
    main_model.fit(X_train[cols], y_train)
    preds = main_model.predict(X_val[cols])
    score = wmape(y_val.values, preds)
    
    all_results.append({
        "Method": method_name, 
        "K": k, 
        "RMSE ($)": np.sqrt(mean_squared_error(y_val, preds)), 
        "WMAPE (%)": score
    })
    
    if score < best_wmape_tracker[method_name]:
        best_wmape_tracker[method_name] = score
        best_predictions[method_name] = preds

# Grid search setup
feature_counts = [3, 4, 5, 6, 7, 8, 9, 10, 11]

# Test loop
for k in feature_counts:
    # MRMR
    mrmr_cols = mrmr_regression(X=X_train, y=y_train, K=k, show_progress=False)
    evaluate_selector("MRMR", mrmr_cols, k)
    
    # RFE
    rfe_sel = RFE(estimator=main_model, n_features_to_select=k).fit(X_train, y_train)
    rfe_cols = X_train.columns[rfe_sel.support_].tolist()
    evaluate_selector("RFE", rfe_cols, k)
    
    # LASSO
    lasso_sel = SelectFromModel(LassoCV(cv=5), max_features=k).fit(X_train, y_train)
    lasso_cols = X_train.columns[lasso_sel.get_support()].tolist()
    evaluate_selector("LASSO", lasso_cols, k)
    
    # SFS
    sfs_sel = SequentialFeatureSelector(LinearRegression(), n_features_to_select=k, direction='forward', n_jobs=-1).fit(X_train, y_train)
    sfs_cols = X_train.columns[sfs_sel.get_support()].tolist()
    evaluate_selector("SFS", sfs_cols, k)

# All features
main_model.fit(X_train, y_train)
pred_all = main_model.predict(X_val)
best_predictions["All Features"] = pred_all
for k in feature_counts:
    all_results.append({"Method": "All Features", "K": len(X_train.columns), "RMSE ($)": np.sqrt(mean_squared_error(y_val, pred_all)), "WMAPE (%)": wmape(y_val.values, pred_all)})

# Create dataframe
exp_df = pd.DataFrame(all_results).round(2)
best_per_algorithm = exp_df.loc[exp_df.groupby("Method")["WMAPE (%)"].idxmin()].sort_values(by="WMAPE (%)").reset_index(drop=True)
print(best_per_algorithm)

 ### Comperative Graphics of Feature Selection Algorithms

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plot_df = pd.DataFrame({'Date': val_data['Date']})

# Load best predictions
for method in best_predictions.keys():
    plot_df[method] = best_predictions[method]

plot_df = plot_df.sort_values(by='Date').reset_index(drop=True)
actual_sales = y_val.values
date_strings = plot_df['Date'].dt.strftime('%Y-%m-%d').tolist()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(date_strings, actual_sales, color='#111111', marker='o', linewidth=3.5, zorder=5, label='Sales')

# Plot results
for method in ["MRMR", "LASSO", "RFE", "SFS", "All Features"]:
    if method in plot_df.columns:
        k_val = int(best_per_algorithm.loc[best_per_algorithm['Method'] == method, 'K'].values[0])
        label_name = f"{method} (K={k_val})" if method != "All Features" else method
        ax.plot(date_strings, plot_df[method], linestyle='--', marker='s', linewidth=1.5, label=label_name)

ax.set_title("Test vs Prediction", fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Weekly Sales ($)", fontsize=12)
plt.xticks(rotation=30, ha='right')
ax.set_ylim(actual_sales.min() * 0.85, actual_sales.max() * 1.15)
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: f"{int(x):,}"))
ax.legend(loc='upper right', frameon=True, fontsize=10)
plt.tight_layout()
plt.show()